<a href="https://colab.research.google.com/github/Sayan-Laha-2005/101-linux-commands/blob/main/PPE_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Step 1 - Installing the packages

In [51]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mustafatayyipbayram/ppe-detection")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'ppe-detection' dataset.
Path to dataset files: /kaggle/input/ppe-detection


In [52]:
!pip install ultralytics

##Step 2 - Importing all the libraries

In [53]:
import ultralytics
ultralytics.checks()

Ultralytics 8.4.25 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 44.7/112.6 GB disk)


In [54]:
from ultralytics import YOLO
from IPython.display import Image

#### Add your Roboflow API Key to Colab Secrets

To securely use your Roboflow API Key without exposing it directly in your notebook, follow these steps:

1.  Click the "🔑 Secrets" icon in the left sidebar of your Colab notebook.
2.  Click "+ New secret".
3.  For the **Name**, type `ROBOFLOW_API_KEY`.
4.  For the **Value**, paste your actual Roboflow API Key.
5.  Ensure the "Notebook access" toggle is turned **ON** for this secret.

Once added, the code below will be able to access your API key securely.

In [55]:
!pip install roboflow

from roboflow import Roboflow
from google.colab import userdata # Import userdata to access Colab secrets

# Retrieve API key from Colab Secrets
api_key = userdata.get("ROBOFLOW_API_KEY")

# Check if API key was retrieved successfully
if api_key is None:
    raise ValueError("ROBOFLOW_API_KEY not found in Colab Secrets. Please add it as instructed above.")

rf = Roboflow(api_key=api_key)

project = rf.workspace("roboflow-universe-projects").project("safety-vests")

version = project.version(14)

dataset = version.download("yolov11")

loading Roboflow workspace...
loading Roboflow project...


In [56]:
dataset.location

'/content/Safety-Vests-14'

In [ ]:
model = YOLO('yolo11n.pt')
model.train(data=f"{dataset.location}/data.yaml", epochs=10, imgsz=640)

Ultralytics 8.4.25 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Safety-Vests-14/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train5, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, persp

To determine the correct path to `data.yaml`, let's list the contents of the downloaded dataset directory.

In [ ]:
import os
print(f"Listing contents of: {dataset.location}")
!ls -F "{dataset.location}"

In [ ]:
Image(r"/content/runs/detect/train2/confusion_matrix.png", width= 800)

In [ ]:
Image(r"/content/runs/detect/train2/labels.jpg", width=600)

In [ ]:
Image("/content/runs/detect/train2/results.png", width=1000)

In [ ]:
!yolo task=detect mode=val model="/content/runs/detect/train2/weights/best.pt" data="/content/Safety-Vests-14/data.yaml"

In [ ]:
!yolo task=detect mode=predict model="/content/runs/detect/train2/weights/best.pt" conf=0.25 source="/content/Safety-Vests-14/test/images" save=True

##Display Prediction Results

In [ ]:
import glob
import os
from IPython.display import Image as IPyImage, display

prediction_folders = glob.glob("/content/runs/detect/predict*")

if prediction_folders:
    latest_folder = max(prediction_folders, key=os.path.getmtime)
    for img in glob.glob(f"{latest_folder}/*.jpg")[1:4]:
        display(IPyImage(filename=img, width=600))
        print("\n")
else:
    print("No prediction folders found in /content/runs/detect/. Please check if the YOLO prediction commands ran successfully.")

##Run Inference using YOLOv11 Model

In [ ]:
import glob

# Find the first image in the test set to use for prediction
test_image_path = glob.glob(f"{dataset.location}/test/images/*")[0]
print(f"Using image for prediction: {test_image_path}")

!yolo task=detect mode=predict model="/content/runs/detect/train2/weights/best.pt" conf=0.25 source="{test_image_path}" save=True

In [ ]:
import os
import glob
from IPython.display import Image

# Assuming test_image_path holds the original image path
# This variable is available from cell 4cef8mmv5-Rj
predicted_image_name = os.path.basename(test_image_path)

# Find the latest prediction folder (e.g., predict, predict2, predict3)
prediction_folders = glob.glob("/content/runs/detect/predict*")
if prediction_folders:
    latest_prediction_folder = max(prediction_folders, key=os.path.getmtime)
    predicted_image_path = os.path.join(latest_prediction_folder, predicted_image_name)
    Image(predicted_image_path, width=600)
else:
    print("No prediction folders found. Please ensure the YOLO prediction command ran successfully.")

In [ ]:
!yolo task=detect mode=predict model="/content/runs/detect/train2/weights/best.pt" source="/content/videoplayback.avi" conf=0.25 save=True

It appears the video file `/content/videoplayback.avi` is missing. Please upload your video file to the Colab environment or provide the correct path to an existing video file.

In [ ]:
# After uploading, update the 'video_path' variable with the correct path to your video file.
video_path = '/content/your_uploaded_video.mp4' # Replace with the actual path to your video file

!yolo task=detect mode=predict model="/content/runs/detect/train/weights/best.pt" source="{video_path}" conf=0.25 save=True

##Compress and Display YOLOv11 Inference Video

In [ ]:
import os
from IPython.display import Video

save_path = "/content/runs/detect/predict3/videoplayback.avi"
compress_path = "/content/videoplayback.mp4"

os.system(f"ffmpeg -y -i {save_path} -vcodec libx264 -acodec aac {compress_path}")

Video(compress_path, embed=True)


In [ ]:
!pip install streamlit

##Zip and Download YOLOv11 Results Folder via Streamlit

In [ ]:
import streamlit as st
import zipfile
import os
from io import BytesIO

def zip_folder(folder_path):
    zip_buffer = BytesIO()
    with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED, compresslevel=9) as zip_file:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, start=folder_path)
                zip_file.write(file_path, arcname)
    zip_buffer.seek(0)
    return zip_buffer

folder_to_zip = "/content/runs"

if os.path.exists(folder_to_zip):
    zipped_folder = zip_folder(folder_to_zip)

    st.success("✅ Folder zipped successfully!")

    st.download_button(
        label="📦 Download Results Folder",
        data=zipped_folder,
        file_name="runs.zip",
        mime="application/zip"
    )
else:
    st.warning(f"⚠ Folder '{folder_to_zip}' not found.")


In [ ]:
model.save('yolo11n_safety_vest_model.pt')
print("Model saved as 'yolo11n_safety_vest_model.pt'")

You can save the trained model using the `model.save()` method. The `train` method automatically saves the `best.pt` and `last.pt` weights in the `runs/detect/train/weights/` directory.

In [ ]:
from google.colab import files
import os

# Define the directory where the weights are saved
weights_dir = '/content/runs/detect/train/weights/'

# Define the file names to download
files_to_download = ['best.pt', 'last.pt']

for filename in files_to_download:
    file_path = os.path.join(weights_dir, filename)
    if os.path.exists(file_path):
        try:
            files.download(file_path)
            print(f"Successfully initiated download for '{filename}'")
        except Exception as e:
            print(f"Error downloading '{filename}': {e}")
    else:
        print(f"Error: File '{filename}' not found at '{file_path}'. Please ensure training completed successfully.")

In [ ]:
from google.colab import files

# Path to the saved model file
model_file_path = 'yolo11n_safety_vest_model.pt'

# Download the model file
try:
    files.download(model_file_path)
    print(f"Successfully initiated download for '{model_file_path}'")
except FileNotFoundError:
    print(f"Error: Model file '{model_file_path}' not found. Please ensure the model was saved correctly.")

In [ ]:
!ls /content/runs/detect/train2/weights


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!cp /content/runs/detect/train2/weights/best.pt /content/drive/MyDrive/best.pt


In [ ]:
from google.colab import files
files.download('/content/runs/detect/train2/weights/best.pt')

In [ ]:
from ultralytics import YOLO
import os

# ১. আপনার ট্রেনিং করা সেরা মডেলটি লোড করুন
# ফাইল পাথে 'train' এর জায়গায় আপনার ফোল্ডারের নাম (যেমন train2) দিন
model_path = '/content/runs/detect/train/weights/best.pt'

if os.path.exists(model_path):
    model = YOLO(model_path)

    # ২. নতুন কোনো ইমেজে প্রেডিকশন রান করুন
    # 'source' এ আপনার আপলোড করা ইমেজের নাম দিন
    results = model.predict(source='safety_vest_test.jpg', save=True, conf=0.5)

    print("Detection complete! Results saved in runs/detect/predict")
else:
    print("Model file not found. Please check the path.")